# `allclose` said False. triton-blackhole finds the bug.

**Runtime → Change runtime type → GPU** then Run all.

Repo: https://github.com/brian-mwirigi/triton-blackhole

In [ ]:
import importlib.util
import re
import subprocess
import sys

import torch
print("cuda", torch.cuda.is_available(), "torch", torch.__version__)

# Pin triton to whatever THIS Colab torch requires (never pip install -U triton).
from importlib.metadata import requires
pinned = None
for r in requires("torch") or []:
    m = re.search(r"triton\s*==\s*([0-9][^\s;]+)", r, re.I)
    if m:
        pinned = m.group(1)
        break
if pinned:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"triton=={pinned}"])

# Prefer PyPI; fall back to git if 0.1.0 not published yet.
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "triton-blackhole"])
except subprocess.CalledProcessError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/brian-mwirigi/triton-blackhole.git",
    ])

import triton_blackhole as tb
print("triton-blackhole", tb.__version__)

## 1. The usual dead end

A fused softmax-like path with **bf16 accum** plus one **bad store mask** on a tile.

In [ ]:
import torch

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
x = torch.randn(32, 256, dtype=torch.bfloat16, device=device)

def torch_ref(x):
    x32 = x.float()
    x32 = x32 - x32.max(dim=-1, keepdim=True).values
    return torch.softmax(x32, dim=-1).to(x.dtype)

def buggy_kernel_sim(x):
    # Stand-in for a Triton kernel: reduce in bf16, then corrupt one tile.
    xl = x.to(torch.bfloat16)
    m = xl.max(dim=-1, keepdim=True).values
    e = torch.exp((xl - m).float()).to(torch.bfloat16)
    out = (e / e.sum(dim=-1, keepdim=True)).to(x.dtype).clone()
    out[7:9, 100:108] = out[7:9, 100:108] + 0.5  # bad mask / bad store
    return out

ref = torch_ref(x)
tri = buggy_kernel_sim(x)

print("torch.allclose →", torch.allclose(tri, ref, atol=1e-2, rtol=1e-2))
print("(that is the entire story… until now)")

## 2. triton-blackhole localizes it

In [ ]:
from triton_blackhole import bisect_axes, classify_drift, compare, format_report
from triton_blackhole.classify import format_classification
from triton_blackhole.triton_hooks import diagnose

print(format_report(compare(tri, ref)))
print()
print(format_classification(classify_drift(tri, ref)))
print()
print(bisect_axes(tri, ref, atol=1e-1, rtol=1e-1).report())
print()
print("--- one-shot diagnose ---")
print(diagnose(tri, ref, atol=1e-1, rtol=1e-1))

## 3. Live Triton kernel (GPU) — should PASS

In [ ]:
assert torch.cuda.is_available(), "Enable GPU runtime"
import triton
import triton.language as tl
from triton_blackhole.triton_hooks import diagnose

@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    tl.store(out_ptr + offs, tl.load(x_ptr + offs, mask=mask) + tl.load(y_ptr + offs, mask=mask), mask=mask)

def triton_add(x, y):
    out = torch.empty_like(x)
    BLOCK = 256
    add_kernel[(triton.cdiv(x.numel(), BLOCK),)](x, y, out, x.numel(), BLOCK=BLOCK)
    return out

a = torch.randn(100_000, device="cuda", dtype=torch.bfloat16)
b = torch.randn_like(a)
print(diagnose(triton_add(a, b), a + b))
print("LIVE TRITON KERNEL OK")